# LogiEdge — Edge AI Cold-Chain Monitoring Pipeline

End-to-end notebook for FreightBridge Logistics' refrigerated-truck monitoring pilot.
Run the cells below **in order**. Each section corresponds to one project component.

**Environment:** Python 3.10, TensorFlow 2.15.0, tensorflow-model-optimization 0.7.5,
Mosquitto broker installed locally.

Before running, activate your virtual environment and install dependencies:
```
pip install -r requirements.txt
```


In [ ]:
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(os.path.join(PROJECT_ROOT, "data_pipeline"))
sys.path.append(os.path.join(PROJECT_ROOT, "training"))

print("Project root:", PROJECT_ROOT)


## Section 1 — Dataset Generation

Generates labelled sensor windows for all three classes (Normal / Warning / Critical) and the
frozen normalisation statistics (`training_stats.npy`), entirely offline — no MQTT broker
required for this step.


In [ ]:
os.chdir(os.path.join(PROJECT_ROOT, "training"))
%run generate_dataset.py
os.chdir(PROJECT_ROOT)


In [ ]:
import pandas as pd

df = pd.read_csv(os.path.join(PROJECT_ROOT, "training", "dataset.csv"))
print(df.shape)
print(df["label"].value_counts().sort_index())
df.head()


## Section 2 — Train the FP32 Baseline (M1)

Trains the 2-hidden-layer MLP (32, 16 units). The script exits with an error if validation
accuracy falls below the required 88% threshold.


In [ ]:
os.chdir(os.path.join(PROJECT_ROOT, "training"))
%run train_model.py
os.chdir(PROJECT_ROOT)


## Section 3 — Convert to INT8 (M2) and Prune + Quantise (M3)

In [ ]:
os.chdir(os.path.join(PROJECT_ROOT, "training"))
%run convert_ptq.py
os.chdir(PROJECT_ROOT)


In [ ]:
os.chdir(os.path.join(PROJECT_ROOT, "training"))
%run prune_quantise.py
os.chdir(PROJECT_ROOT)


## Section 4 — Arithmetic Intensity and Roofline (Task B2)

Uses the figures given in the brief: 45 MFLOPs/inference, 18 MB accessed/inference,
CPU peak 16 GFLOP/s, bandwidth 12 GB/s.


In [ ]:
flops = 45e6
bytes_accessed = 18 * 1024 * 1024
peak_gflops = 16.0
peak_bandwidth_gbs = 12.0

arithmetic_intensity = flops / bytes_accessed          # FLOPs per byte
ridge_point = (peak_gflops * 1e9) / (peak_bandwidth_gbs * 1e9)  # FLOPs per byte

print(f"Arithmetic Intensity: {arithmetic_intensity:.4f} FLOPs/byte")
print(f"Ridge point:          {ridge_point:.4f} FLOPs/byte")

if arithmetic_intensity < ridge_point:
    print("Classification: MEMORY-BANDWIDTH BOUND")
    print("Implication: reducing bytes moved per inference (INT8 quantisation, pruning) "
          "improves latency more than adding raw compute.")
else:
    print("Classification: COMPUTE BOUND")
    print("Implication: reducing FLOPs (smaller architecture) improves latency more than "
          "reducing memory traffic.")


## Section 5 — Five-Metric Benchmarking and Pareto Analysis (Module 6)

Benchmarks M1, M2, and M3 on latency (mean/p95), model size, accuracy, Class 2 recall, and
estimated energy per inference. Produces `optimisation/results/benchmark_results.csv` and
`pareto_chart.png`.


In [ ]:
os.chdir(os.path.join(PROJECT_ROOT, "optimisation"))
%run benchmark.py
os.chdir(PROJECT_ROOT)


In [ ]:
from IPython.display import Image, display

results_csv = os.path.join(PROJECT_ROOT, "optimisation", "results", "benchmark_results.csv")
results_df = pd.read_csv(results_csv)
display(results_df)

pareto_png = os.path.join(PROJECT_ROOT, "optimisation", "results", "pareto_chart.png")
display(Image(filename=pareto_png))


## Section 6 — Live MQTT Demo (run outside the notebook)

The simulator, inference service, and drift monitor are long-running processes and are best run
as standalone terminal processes rather than notebook cells. From a terminal at the project root:

```
# Terminal A
mosquitto -v

# Terminal B
set MODEL_PATH=training\models\model.tflite
set STATS_PATH=data_pipeline\training_stats.npy
set BROKER_HOST=localhost
set TRUCK_ID=TRK-001
python inference\inference_service.py

# Terminal C
python data_pipeline\simulator.py --anomaly none --truck-id TRK-001 --duration 300
```

Repeat Terminal C with `--anomaly temp_drift`, `--anomaly vibration`, and `--anomaly combined`
to observe all three classes trigger in Terminal B's console output.

See `README.md` Section 4, Steps 6-9 for the drift-monitor, Docker, and Ansible demo sequences.


## Section 7 — Notes for the Final Report

Pull these values directly from the cells above into your Final Report:

- **Section 1 (Deployment Context):** dataset class counts from Section 1; Arithmetic Intensity
  and Roofline classification from Section 4.
- **Section 4 (Optimisation and Pareto Analysis):** the `benchmark_results.csv` table and
  `pareto_chart.png` from Section 5, plus the Class 2 (Critical) recall column — the recommended
  variant must exceed 95% recall on Critical per Task F3.
